In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('../data/customer_transactions.csv')

display(df.head())

,date,store_nbr,lylty_card_nbr,txn_id,prod_nbr,prod_qty,tot_sales,clean_prod_name,net_weight,brand,lifestage,premium_customer
0,2018-10-17,1,1000,1,5,2,6.0,natural chip compny seasalt 175 g,175,Natural Chip Company,YOUNG SINGLES/COUPLES,Premium
1,2019-05-14,1,1307,348,66,3,6.3,ccs nacho cheese 175 g,175,CCs,MIDAGE SINGLES/COUPLES,Budget
2,2019-05-20,1,1343,383,61,2,2.9,smiths crinkle cut chips chicken 170 g,170,Smith's,MIDAGE SINGLES/COUPLES,Budget
3,2018-08-17,2,2373,974,69,5,15.0,smiths chip thinly s cream onion 175 g,175,Smith's,MIDAGE SINGLES/COUPLES,Budget
4,2018-08-18,2,2426,1038,108,3,13.8,kettle tortilla chpshny jlpno chili 150 g,150,Kettle,MIDAGE SINGLES/COUPLES,Budget


In [ ]:
# total sales and sales % per brand, across the entire population

brand_sales = df.groupby("brand")["tot_sales"].sum()

top_brands = (
    brand_sales.nlargest(10)
    .reset_index(name="total_sales")
)
top_brands["sales_share_%"] = top_brands["total_sales"] / brand_sales.sum() * 100

# tidy up the format to show 2 decimal places and a percentage sign
top_brands["sales_share_%"] = top_brands["sales_share_%"].map(lambda v: f"{v:.2f}%")

print(top_brands)

           brand  total_sales sales_share_%
0         Kettle     390239.8        21.70%
1        Doritos     226329.9        12.59%
2        Smith's     210440.0        11.70%
3       Pringles     177655.5         9.88%
4      Infuzions      99047.6         5.51%
5          Thins      88852.5         4.94%
6  Red Rock Deli      87607.5         4.87%
7       Twisties      81522.1         4.53%
8       Tostitos      79789.6         4.44%
9           Cobs      70569.8         3.92%


**Kettle is the standount brand** <br>
- accounting for a fifth of all sales, the strongest performance out of 20 brands<br>
<br>
**Dorito's, Smith's and Pringles are reliable, accounting for approx. 10-13% each** <br>
<br>
- these are the most well-received brands across the entire population. this is a great starting point and would help isolate top brands <br>
- positioning items from these brands on featured sections, and near the checkout isle would take advantage on their widespread appeal. <br>

In [4]:
# total sales, sales %, and unique products per brand, across the entire population

brand_summary = df.groupby("brand").agg(
    total_sales=("tot_sales", "sum"),
    unique_products=("clean_prod_name", "nunique"),
)

all_sales = brand_summary["total_sales"].sum()

top_brands = brand_summary.nlargest(10, "total_sales").reset_index()
top_brands["sales_share_%"] = top_brands["total_sales"] / all_sales * 100

top_brands = top_brands[["brand", "total_sales", "sales_share_%", "unique_products"]]

# tidy up the format to show 2 decimal places and a percentage sign
top_brands["sales_share_%"] = top_brands["sales_share_%"].map(lambda v: f"{v:.2f}%")

print(top_brands)

           brand  total_sales sales_share_%  unique_products
0         Kettle     390239.8        21.70%               13
1        Doritos     226329.9        12.59%                8
2        Smith's     210440.0        11.70%               16
3       Pringles     177655.5         9.88%                8
4      Infuzions      99047.6         5.51%                5
5          Thins      88852.5         4.94%                5
6  Red Rock Deli      87607.5         4.87%               11
7       Twisties      81522.1         4.53%                3
8       Tostitos      79789.6         4.44%                3
9           Cobs      70569.8         3.92%                3


the disparity on sales is not solely due to an imbalance of product offerings. high sales% is due to customer preference.

In [ ]:
# create variables for each of the five target cohorts


# budget older families
segment_b_olderfamilies = df[
    (df["lifestage"] == "OLDER FAMILIES") & (df["premium_customer"] == "Budget")
]

# budget young families
segment_b_youngfamilies = df[
    (df["lifestage"] == "YOUNG FAMILIES") & (df["premium_customer"] == "Budget")
]


# mainstream young singles/couples
segment_main_youngsc = df[
    (df["lifestage"] == "YOUNG SINGLES/COUPLES") & (df["premium_customer"] == "Mainstream")
]

# older singles/couples
segment_older_sc = df[df["lifestage"] == "OLDER SINGLES/COUPLES"] #includes all 3 membership tiers


# mainstream retirees
segment_main_retirees = df[
    (df["lifestage"] == "RETIREES") & (df["premium_customer"] == "Mainstream")
]

In [6]:
# top 10 brands for each of the five target cohorts

target_cohorts = {
    "Budget Older Families":        segment_b_olderfamilies,
    "Budget Young Families":        segment_b_youngfamilies,
    "Mainstream Young Singles/Couples": segment_main_youngsc,
    "Older Singles/Couples":        segment_older_sc,
    "Mainstream Retirees":          segment_main_retirees,
}

def top_brands_for(segment, n=10):
    summary = segment.groupby("brand").agg(
        total_sales=("tot_sales", "sum"),
        unique_products=("clean_prod_name", "nunique"),
    )
    cohort_sales = summary["total_sales"].sum()

    top = summary.nlargest(n, "total_sales").reset_index()
    top["sales_share_%"] = top["total_sales"] / cohort_sales * 100
    top.insert(0, "rank", range(1, len(top) + 1))
    return top[["rank", "brand", "total_sales", "sales_share_%", "unique_products"]]

top_brands_by_cohort = pd.concat(
    {name: top_brands_for(seg) for name, seg in target_cohorts.items()},
    names=["cohort"],
).reset_index(level="cohort").reset_index(drop=True)

# tidy up the format to show 2 decimal places and a percentage sign
top_brands_by_cohort["sales_share_%"] = top_brands_by_cohort["sales_share_%"].map(lambda v: f"{v:.2f}%")

display(top_brands_by_cohort)

,cohort,rank,brand,total_sales,sales_share_%,unique_products
0,Budget Older Families,1,Kettle,32058.00,20.54%,13
1,Budget Older Families,2,Smith's,20150.90,12.91%,16
2,Budget Older Families,3,Doritos,18555.55,11.89%,8
3,Budget Older Families,4,Pringles,14300.50,9.16%,8
4,Budget Older Families,5,Red Rock Deli,9379.20,6.01%,11
5,Budget Older Families,6,Infuzions,8365.00,5.36%,5
6,Budget Older Families,7,Thins,7517.40,4.82%,5
7,Budget Older Families,8,Twisties,7164.60,4.59%,3
8,Budget Older Families,9,Tostitos,6006.00,3.85%,3
9,Budget Older Families,10,Cobs,5658.20,3.62%,3


quite a long list compiling the top10 brands for each target segment, but with familiar names throughout

- Kettle consistently holds the top spot, at an approximate avg of 20% sales_share
- Smith's, Doritos, and Pringles make up the rest of the top 4.

*let's narrow down the results to highlight the standout brands*

In [10]:
top_brands_by_cohort = pd.concat(
    {name: top_brands_for(seg) for name, seg in target_cohorts.items()},
    names=["target_cohorts"],
).reset_index(level="target_cohorts").reset_index(drop=True)

# keep only brands with a sales share above 10% (filter before formatting as text)
top_brands_by_cohort = top_brands_by_cohort[top_brands_by_cohort["sales_share_%"] > 10]

# tidy up the format to show 2 decimal places and a percentage sign
top_brands_by_cohort["sales_share_%"] = top_brands_by_cohort["sales_share_%"].map(lambda v: f"{v:.2f}%")

display(top_brands_by_cohort)

,target_cohorts,rank,brand,total_sales,sales_share_%,unique_products
0,Budget Older Families,1,Kettle,32058.00,20.54%,13
1,Budget Older Families,2,Smith's,20150.90,12.91%,16
2,Budget Older Families,3,Doritos,18555.55,11.89%,8
10,Budget Young Families,1,Kettle,26369.60,20.42%,13
11,Budget Young Families,2,Smith's,16171.80,12.52%,16
12,Budget Young Families,3,Doritos,16078.05,12.45%,8
20,Mainstream Young Singles/Couples,1,Kettle,35423.60,24.06%,13
21,Mainstream Young Singles/Couples,2,Doritos,20925.90,14.21%,8
22,Mainstream Young Singles/Couples,3,Pringles,16006.20,10.87%,8
30,Older Singles/Couples,1,Kettle,83862.60,22.39%,13


the following brands should be prioritized on top of shelves, and in appealing to the target cohorts. <br>

with these out of the way, let's look at the combined averages of the other part of the top 10. these are <10% in share. 

In [12]:
# top 10 brands per cohort (numeric shares, before any filtering or formatting)
all_top10 = pd.concat(
    {name: top_brands_for(seg) for name, seg in target_cohorts.items()},
    names=["target_cohorts"],
).reset_index(level="target_cohorts").reset_index(drop=True)

# keep only top-10 brands with a sales share below 10% in that cohort
below_10 = all_top10[all_top10["sales_share_%"] < 10]

# one row per unique brand, averaging its sales share across the cohorts it appears in
brand_avg_share = (
    below_10.groupby("brand")
    .agg(
        avg_sales_share=("sales_share_%", "mean"),
        n_cohorts=("target_cohorts", "nunique"),
    )
    .sort_values("avg_sales_share", ascending=False)
    .reset_index()
)

# tidy up the format to show 2 decimal places and a percentage sign
brand_avg_share["avg_sales_share"] = brand_avg_share["avg_sales_share"].map(lambda v: f"{v:.2f}%")

display(brand_avg_share)

,brand,avg_sales_share,n_cohorts
0,Smith's,9.93%,1
1,Pringles,9.25%,2
2,Infuzions,5.58%,5
3,Red Rock Deli,5.15%,4
4,Thins,4.93%,5
5,Twisties,4.63%,5
6,Tostitos,4.40%,5
7,Cobs,3.83%,5
8,Tyrrells,3.26%,1


the following brands belong in at least 1 cohort's top 10, each accounting in sales % lower than 10%

- these are great secondary cluster of brands to prioritize in the store layout for all chip buyers. 

- brands from index [2-7] are the most consistent, as their scores are aggregated from 4-5 target cohorts.